# Homework 1 — Data Aggregation

Merges `train`/`test` with `building_metadata` and `weather` into one table each (see [EDA.ipynb](../eda/EDA.ipynb)).

Steps:

1. Convert site-0 electricity from kBTU to kWh.
2. Weather: drop `cloud_coverage` and `precip_depth_1_hr`, shift from UTC to local time, fill gaps.
3. Left-join `train`/`test` ← `building_metadata` ← `weather`, add `local_timestamp`.
4. Flag anomalies in train.
5. Save to Parquet.


In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "..")  # the ashrae package lives in the homework folder
from ashrae.config import KBTU_TO_KWH, METERS, PROCESSED, WEATHER_SHIFT_HOURS
from ashrae.data import anomaly_flags, clean_weather, read_building, read_readings, read_weather

PROCESSED.mkdir(exist_ok=True)

# cloud_coverage and precip_depth_1_hr are dropped here: they're left out of this list,
# so the weather files are read without them
WEATHER_COLS = [
    "air_temperature",
    "dew_temperature",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed",
]

In [2]:
building = read_building()
train, test = read_readings("train.csv"), read_readings("test.csv")
weather_train = read_weather("weather_train.csv", WEATHER_COLS)
weather_test = read_weather("weather_test.csv", WEATHER_COLS)

## 1. Site-0 electricity: kBTU → kWh

Per the data description, site 0 electricity readings are in kBTU. Predictions for these rows must be divided
by the same factor before submission.


In [3]:
site0 = building.loc[building.site_id == 0, "building_id"]
mask = train.building_id.isin(site0) & (train.meter == 0)
train.loc[mask, "meter_reading"] *= KBTU_TO_KWH
print(f"converted {mask.sum():,} rows")

converted 908,409 rows


## 2. Weather

- `cloud_coverage` and `precip_depth_1_hr` are dropped: often missing, and little or no correlation with the target
  (EDA §7).
- Weather timestamps are UTC, meter timestamps are local (EDA §6.1). Weather is shifted by each site's UTC offset,
  except at site 14, whose meters are in UTC too.
- Train and test weather are cleaned as one table, so the hours around the train/test boundary use real weather
  after the shift instead of copies of the last value.
- Some (site, hour) rows are absent, not just NaN, so each site is first reindexed onto a full hourly grid.
- Gaps are filled by linear interpolation within each site.
- A site with no data at all for a column (site 5 `sea_level_pressure`) gets the mean of the other sites at the
  same hour. Pressure and wind come from weather systems that span large areas, so other sites are a reasonable
  proxy.
- `wind_direction` is interpolated as sin/cos so that 350° → 10° goes through 0°, not 180°.


In [4]:
# local time = UTC + offset (EDA §6.1); WEATHER_SHIFT_HOURS in ashrae/config.py leaves site 14 unshifted,
# because its meters are in UTC too
weather = pd.concat([weather_train, weather_test], ignore_index=True)
weather["timestamp"] += pd.to_timedelta(weather.site_id.map(WEATHER_SHIFT_HOURS), unit="h")
weather_clean = clean_weather(weather)  # grid, interpolation and fallbacks as listed above; in ashrae/data.py

pd.DataFrame(
    {
        "% missing before": weather[WEATHER_COLS].isna().mean() * 100,
        "% missing after": weather_clean[WEATHER_COLS].isna().mean() * 100,
    }
).round(1)

,% missing before,% missing after
air_temperature,0.0,0.0
dew_temperature,0.1,0.0
sea_level_pressure,7.6,0.0
wind_direction,4.5,0.0
wind_speed,0.2,0.0


In [5]:
print(len(weather_clean) - len(weather))

3960


## 3. Merge

Left joins from `train`/`test`, so no target rows are lost.

`local_timestamp` is the timestamp to derive hour and day-of-week features from. It equals `timestamp` everywhere
except at site 14, whose meter timestamps are UTC, so local time is 5 hours earlier. `timestamp` itself stays as
given, since it's the key to `test.csv` and the submission.


In [6]:
def merge(base: pd.DataFrame) -> pd.DataFrame:
    out = base.merge(building, on="building_id", how="left").merge(
        weather_clean, on=["site_id", "timestamp"], how="left"
    )
    assert len(out) == len(base)
    out["local_timestamp"] = out.timestamp - pd.to_timedelta((out.site_id == 14) * 5, unit="h")
    return out


train_merged = merge(train)
test_merged = merge(test)

print("train", train_merged.shape, "| test", test_merged.shape)
display(
    train_merged.isna()
    .mean()
    .mul(100)
    .round(2)
    .loc[lambda s: s > 0]
    .to_frame("% missing")
)
display(
    test_merged.isna()
    .mean()
    .mul(100)
    .round(2)
    .loc[lambda s: s > 0]
    .to_frame("% missing")
)

train (20216100, 15) | test (41697600, 15)


,% missing
year_built,59.99
floor_count,82.65


,% missing
year_built,58.99
floor_count,82.61


In [7]:
train_merged.head()

,building_id,meter,timestamp,meter_reading,site_id,primary_use,square_feet,year_built,floor_count,air_temperature,dew_temperature,sea_level_pressure,wind_speed,wind_direction,local_timestamp
0,0,0,2016-01-01,0.0,0,Education,7432,2008.0,NaN,19.4,19.4,1019.700012,0.0,0.0,2016-01-01
1,1,0,2016-01-01,0.0,0,Education,2720,2004.0,NaN,19.4,19.4,1019.700012,0.0,0.0,2016-01-01
2,2,0,2016-01-01,0.0,0,Education,5376,1991.0,NaN,19.4,19.4,1019.700012,0.0,0.0,2016-01-01
3,3,0,2016-01-01,0.0,0,Education,23685,2002.0,NaN,19.4,19.4,1019.700012,0.0,0.0,2016-01-01
4,4,0,2016-01-01,0.0,0,Education,116607,1975.0,NaN,19.4,19.4,1019.700012,0.0,0.0,2016-01-01


## 4. Anomalies

`anomaly` marks train rows that don't reflect energy use ([EDA §9](../eda/EDA.ipynb#broken)):

- every zero electricity reading: a building always draws some power;
- steam or hot water at 0 for at least a week with the week's mean temperature below 10 °C; chilled water the same
  above 20 °C.

Rows are flagged, not dropped: drop them from training, keep them when scoring validation. LightGBM, 4-fold average
([EDA §9](../eda/EDA.ipynb#broken)): 1.140 → 1.048 on clean rows, 1.286 → 1.281 on all rows.

In [8]:
flags = anomaly_flags(train_merged)  # the rules of EDA §9, in ashrae/data.py
train_merged["anomaly"] = flags.zero_electricity | flags.off_when_needed
print(f"anomalies: {train_merged.anomaly.sum():,} rows ({train_merged.anomaly.mean():.1%} of train)")
train_merged.groupby(train_merged.meter.map(METERS)).anomaly.mean().mul(100).round(1).rename("% flagged")

anomalies: 599,086 rows (3.0% of train)


meter
chilledwater    1.0
electricity     4.4
hotwater        0.4
steam           0.8
Name: % flagged, dtype: float64

## 5. Save


In [9]:
train_merged.to_parquet(PROCESSED / "train_merged.parquet", index=False)
test_merged.to_parquet(PROCESSED / "test_merged.parquet", index=False)

## Next steps

- Target: train on `log1p(meter_reading)` — RMSE on it equals RMSLE.
- Drop `anomaly` rows from training; keep them when scoring validation.
- Time features (hour, day of week, holidays): derive from `local_timestamp`, not `timestamp`.
- `square_feet`: use `log1p`.
- `year_built` (60% missing) and `floor_count` (83% missing): add a missing flag, or drop.
- Fit any imputation statistics on train only.
